# Term-Deposit Lead Scoring Service

## 1. Purpose of the project

Build a small, production-style ML service end to end: train a model that scores
bank customers, then serve it the way real AI-infrastructure teams do, using
**ONNX** (portable model format), **NVIDIA Triton** (model server), **FastAPI**
(API layer) and **vLLM** (LLM that explains each score), all in containers,
with a benchmark of speed and throughput.

The model itself is deliberately simple. The goal of the project is **serving and
systems skills**: packaging a model, running it behind an inference server,
and measuring latency and throughput.

## 2. Dataset

- **Name:** Bank Marketing (UCI Machine Learning Repository, dataset id 222)
- **Link:** https://archive.ics.uci.edu/dataset/222/bank+marketing
- **Source:** Moro, Cortez and Rita (2014), phone marketing campaigns of a
  Portuguese bank
- **File used:** `bank-full.csv`
- **Format:** semicolon-separated (`sep=";"`)
- **Size:** 45,211 rows (one row per customer contact) x 17 columns

## 3. About the data

Each row is one customer who was phoned about a **term deposit**
(a savings product with a fixed term and a fixed rate).

| Group | Columns | Meaning |
|---|---|---|
| Customer profile | `age`, `job`, `marital`, `education` | Who the customer is |
| Financial status | `default`, `balance`, `housing`, `loan` | Credit default flag, average yearly balance, housing loan, personal loan |
| Current contact | `contact`, `day`, `month`, `duration` | How and when the customer was last contacted; call length in seconds |
| Campaign history | `campaign`, `pdays`, `previous`, `poutcome` | Contacts in this campaign, days since last contact (-1 = never), earlier contacts, outcome of the earlier campaign |
| **Target** | `y` | Did the customer subscribe to the term deposit? (yes / no) |

Key facts found during data checks:
- **No missing values** (0 nulls).
- **Imbalanced target:** about 88.3% "no" and 11.7% "yes".
- Some columns use `"unknown"` as a category (`poutcome` 36,959, `contact` 13,020,
  `education` 1,857, `job` 288). We keep it as its own category instead of dropping rows.
- **`duration` is a leaky feature.** Call length is only known after the call ends,
  and long calls almost always mean "yes". We drop it so the model only uses
  information available **before** a call.

## 4. Problem statement

Calling every customer is expensive, and only about 1 in 8 says yes.
**Can we predict, before the call, which customers are most likely to subscribe,
so the bank can call the best prospects first?**

This is a **binary classification** problem: given a customer's profile and history,
predict the probability of `y = yes`.

## 5. How we are solving it

1. **Train** a scikit-learn `GradientBoostingClassifier` on the customer features
   (without `duration`).
2. **Evaluate** with ROC AUC, precision and recall, not just accuracy, because the
   classes are imbalanced (a model that always says "no" is already 88% accurate).
3. **Tune the decision threshold** (instead of the default 0.5) to balance
   precision and recall.
4. **Export to ONNX**, a portable format, and check it matches the original model.
5. **Serve with NVIDIA Triton** in Docker, using the ONNX Runtime backend with
   dynamic batching.
6. **Wrap it with FastAPI**, which accepts a customer's JSON, calls Triton and
   applies the threshold.
7. **Explain each score with vLLM**, which writes a plain-English reason.
8. **Benchmark** latency (p50/p95) and requests per second, with and without batching.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import pandas as pd

path = "/content/drive/MyDrive/UCI Bank Marketing dataset/bank/bank-full.csv"
df = pd.read_csv(path, sep=";")
print(df.shape)
df.head()

(45211, 17)


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no


In [3]:
print(df.isna().sum().sum())                      # missing values, expect 0
print(df["y"].value_counts(normalize=True))       # expect about 88% no, 12% yes
print((df == "unknown").sum())                    # columns that use "unknown" as a category

0
y
no     0.883015
yes    0.116985
Name: proportion, dtype: float64
age              0
job            288
marital          0
education     1857
default          0
balance          0
housing          0
loan             0
contact      13020
day              0
month            0
duration         0
campaign         0
pdays            0
previous         0
poutcome     36959
y                0
dtype: int64


1. **Train** a scikit-learn `GradientBoostingClassifier` on the customer features
   (without `duration`). Tree-based boosting is a strong default for tabular data,
   needs no feature scaling, captures feature interactions automatically, and
   exports cleanly to ONNX for Triton. We compare it against logistic regression,
   random forest and HistGradientBoosting to confirm the choice.

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import classification_report, roc_auc_score

X = df.drop(columns=["y", "duration"])            # drop duration: it leaks the answer
y = (df["y"] == "yes").astype(int)

X = pd.get_dummies(X).astype("float32")           # one-hot encode the text columns
feature_names = list(X.columns)                   # keep this; the API will need the same order

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

model = GradientBoostingClassifier(random_state=42)
model.fit(X_train.values, y_train)

proba = model.predict_proba(X_test.values)[:, 1]
print(classification_report(y_test, proba > 0.5))
print("ROC AUC:", roc_auc_score(y_test, proba))

              precision    recall  f1-score   support

           0       0.90      0.99      0.94      7985
           1       0.66      0.20      0.31      1058

    accuracy                           0.89      9043
   macro avg       0.78      0.59      0.62      9043
weighted avg       0.87      0.89      0.87      9043

ROC AUC: 0.8021150834563389


In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

candidates = {
    "Logistic Regression (baseline)": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "Random Forest": RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42),
    "HistGradientBoosting": HistGradientBoostingClassifier(random_state=42),
    "GradientBoosting": model,
}

for name, m in candidates.items():
    if m is not model:
        m.fit(X_train.values, y_train)
    p = m.predict_proba(X_test.values)[:, 1]
    print(f"{name:32s} ROC AUC = {roc_auc_score(y_test, p):.3f}")

Logistic Regression (baseline)   ROC AUC = 0.772
Random Forest                    ROC AUC = 0.792
HistGradientBoosting             ROC AUC = 0.803
GradientBoosting (ours)          ROC AUC = 0.802


### Why GradientBoosting? Comparing against alternatives

We train three other models on the same train/test split and compare ROC AUC.
We keep GradientBoosting if it is close to the best, because it also exports
cleanly to ONNX for Triton.

In [6]:
from sklearn.metrics import precision_recall_curve

prec, rec, thr = precision_recall_curve(y_test, proba)
f1 = 2 * prec[:-1] * rec[:-1] / (prec[:-1] + rec[:-1] + 1e-12)
best = f1.argmax()
threshold = float(thr[best])
print(f"threshold={threshold:.3f}  precision={prec[best]:.2f}  recall={rec[best]:.2f}  F1={f1[best]:.2f}")
print(classification_report(y_test, proba >= threshold))

threshold=0.178  precision=0.47  recall=0.53  F1=0.50
              precision    recall  f1-score   support

           0       0.94      0.92      0.93      7985
           1       0.47      0.53      0.50      1058

    accuracy                           0.88      9043
   macro avg       0.70      0.72      0.71      9043
weighted avg       0.88      0.88      0.88      9043



In [14]:
!pip -q install skl2onnx onnxruntime

import os, json, shutil
import numpy as np
import onnxruntime as ort
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType

n_features = X_train.shape[1]
onx = convert_sklearn(
    model,
    initial_types=[("input", FloatTensorType([None, n_features]))],
    options={id(model): {"zipmap": False}},
    target_opset={"": 15, "ai.onnx.ml": 3},   # pinned so Triton's ONNX Runtime accepts it
)

repo = "model_repository/lead_scorer"
os.makedirs(f"{repo}/1", exist_ok=True)
with open(f"{repo}/1/model.onnx", "wb") as f:
    f.write(onx.SerializeToString())

# parity check: ONNX output must match sklearn
sess = ort.InferenceSession(f"{repo}/1/model.onnx")
out = sess.run(None, {"input": X_test.values[:500]})
onnx_proba = out[1][:, 1]
print("max abs diff vs sklearn:", np.abs(onnx_proba - proba[:500]).max())

config = f'''name: "lead_scorer"
backend: "onnxruntime"
max_batch_size: 64
input [
  {{ name: "input" data_type: TYPE_FP32 dims: [ {n_features} ] }}
]
output [
  {{ name: "probabilities" data_type: TYPE_FP32 dims: [ 2 ] }}
]
dynamic_batching {{
  preferred_batch_size: [ 8, 16, 32 ]
  max_queue_delay_microseconds: 5000
}}
instance_group [ {{ kind: KIND_CPU count: 2 }} ]
'''
with open(f"{repo}/config.pbtxt", "w") as f:
    f.write(config)

with open("model_repository/metadata.json", "w") as f:
    json.dump({"feature_names": feature_names, "threshold": threshold}, f)

shutil.make_archive("model_repository", "zip", ".", "model_repository")
from google.colab import files
files.download("model_repository.zip")

max abs diff vs sklearn: 3.893368655250029e-07


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>